
 01_data_exploration.ipynb - Complete Code

In [ ]:

# ============================================================
# 🎌 ANIME FACE GAN - NOTEBOOK 01: DATA EXPLORATION
# ============================================================
# Author: Vineet Yadav
# Platform: Kaggle (T4 GPU)
# Dataset: Anime Face Dataset (63,632 images)
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")
print(f"📁 Working directory: {os.getcwd()}")

In [ ]:

# ============================================================
# LOCATE DATASET - FIXED VERSION
# ============================================================
import os

DATASET_PATH = None

# All possible paths to check
search_paths = [
    '/kaggle/input/animefacedataset',
    '/kaggle/input/animefacedataset/images',
    '/kaggle/input/anime-face-dataset',
    '/kaggle/input/anime-face-dataset/images',
    '/kaggle/input/animefacedataset/data',
    '/kaggle/input/animefaces',
    '/kaggle/input/animefacedataset/animefacedataset',
]

# Auto search through all paths
print("🔍 Checking known paths...")
for path in search_paths:
    if os.path.exists(path):
        contents = os.listdir(path)
        # Check if this folder has images
        image_exts = {'.jpg', '.jpeg', '.png'}
        has_images = any(
            os.path.splitext(f)[1].lower() in image_exts 
            for f in contents
        )
        if has_images:
            DATASET_PATH = path
            print(f"✅ Dataset found at: {DATASET_PATH}")
            print(f"📊 Sample files: {contents[:5]}")
            break
        else:
            print(f"📂 Found folder but no images: {path}")
            print(f"   Contents: {contents[:5]}")

# If still not found, do deep search
if DATASET_PATH is None:
    print("\n🔍 Doing deep search in /kaggle/input/...")
    for root, dirs, files in os.walk('/kaggle/input/'):
        image_files = [
            f for f in files 
            if os.path.splitext(f)[1].lower() in {'.jpg', '.jpeg', '.png'}
        ]
        if len(image_files) > 100:  # Found folder with many images
            DATASET_PATH = root
            print(f"✅ Images found at: {DATASET_PATH}")
            print(f"📊 Image count in folder: {len(image_files)}")
            print(f"📋 Sample files: {image_files[:5]}")
            break

if DATASET_PATH is None:
    print("\n❌ Dataset still not found!")
    print("👉 Please follow steps below to add dataset")
else:
    print(f"\n✅ SUCCESS! Use this path: {DATASET_PATH}")

In [ ]:

# ============================================================
# COUNT TOTAL IMAGES
# ============================================================

image_extensions = {'.jpg', '.jpeg', '.png', '.gif', '.bmp', '.webp'}
all_image_paths = []

for root, dirs, files in os.walk(DATASET_PATH):
    for file in files:
        ext = os.path.splitext(file)[1].lower()
        if ext in image_extensions:
            all_image_paths.append(os.path.join(root, file))

print(f"📊 DATASET STATISTICS")
print(f"{'='*40}")
print(f"✅ Total Images Found : {len(all_image_paths)}")
print(f"📁 Dataset Path       : {DATASET_PATH}")
print(f"🖼️  File Extensions    : {image_extensions}")
print(f"{'='*40}")

In [ ]:

# ============================================================
# DISPLAY SAMPLE IMAGE GRID (5x5 = 25 images)
# ============================================================

import random
random.seed(42)
sample_paths = random.sample(all_image_paths, min(25, len(all_image_paths)))

fig, axes = plt.subplots(5, 5, figsize=(12, 12))
fig.suptitle('🎌 Sample Anime Face Images', fontsize=18, fontweight='bold', y=1.02)

for idx, ax in enumerate(axes.flatten()):
    try:
        img = Image.open(sample_paths[idx]).convert('RGB')
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f'#{idx+1}', fontsize=8)
    except Exception as e:
        ax.axis('off')
        ax.set_title('Error', fontsize=8)

plt.tight_layout()
plt.savefig('/kaggle/working/sample_grid.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Sample grid saved!")

In [ ]:

# ============================================================
# IMAGE SIZE ANALYSIS
# ============================================================

print("🔍 Analyzing image sizes... (sampling 2000 images)")
sample_for_analysis = random.sample(all_image_paths, min(2000, len(all_image_paths)))

widths, heights = [], []
formats = {}
corrupted = []

for path in sample_for_analysis:
    try:
        with Image.open(path) as img:
            w, h = img.size
            widths.append(w)
            heights.append(h)
            fmt = img.format if img.format else 'Unknown'
            formats[fmt] = formats.get(fmt, 0) + 1
    except Exception:
        corrupted.append(path)

print(f"\n📐 IMAGE SIZE STATISTICS (from {len(sample_for_analysis)} samples)")
print(f"{'='*45}")
print(f"Width  → Min: {min(widths)}px | Max: {max(widths)}px | Avg: {np.mean(widths):.1f}px")
print(f"Height → Min: {min(heights)}px | Max: {max(heights)}px | Avg: {np.mean(heights):.1f}px")
print(f"\n📄 File Formats: {formats}")
print(f"❌ Corrupted in sample: {len(corrupted)}")
print(f"{'='*45}")

In [ ]:

# ============================================================
# SIZE DISTRIBUTION PLOT
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('📐 Image Size Distribution', fontsize=14, fontweight='bold')

axes[0].hist(widths, bins=30, color='#FF6B9D', edgecolor='black', alpha=0.8)
axes[0].set_title('Width Distribution')
axes[0].set_xlabel('Width (pixels)')
axes[0].set_ylabel('Count')
axes[0].axvline(np.mean(widths), color='red', linestyle='--', label=f'Mean: {np.mean(widths):.0f}px')
axes[0].legend()

axes[1].hist(heights, bins=30, color='#9B59B6', edgecolor='black', alpha=0.8)
axes[1].set_title('Height Distribution')
axes[1].set_xlabel('Height (pixels)')
axes[1].set_ylabel('Count')
axes[1].axvline(np.mean(heights), color='red', linestyle='--', label=f'Mean: {np.mean(heights):.0f}px')
axes[1].legend()

plt.tight_layout()
plt.savefig('/kaggle/working/size_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Size distribution plot saved!")

In [ ]:

# ============================================================
# COLOR DISTRIBUTION ANALYSIS
# ============================================================

print("🎨 Analyzing color distributions... (sampling 500 images)")
color_sample = random.sample(all_image_paths, min(500, len(all_image_paths)))

r_means, g_means, b_means = [], [], []

for path in color_sample:
    try:
        with Image.open(path).convert('RGB') as img:
            img_resized = img.resize((64, 64))
            img_array = np.array(img_resized)
            r_means.append(img_array[:, :, 0].mean())
            g_means.append(img_array[:, :, 1].mean())
            b_means.append(img_array[:, :, 2].mean())
    except Exception:
        pass

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('🎨 RGB Color Channel Distribution', fontsize=14, fontweight='bold')

axes[0].hist(r_means, bins=30, color='red', alpha=0.7, edgecolor='black')
axes[0].set_title('Red Channel')
axes[0].set_xlabel('Mean Pixel Value')
axes[0].set_ylabel('Count')
axes[0].axvline(np.mean(r_means), color='darkred', linestyle='--', label=f'Mean: {np.mean(r_means):.1f}')
axes[0].legend()

axes[1].hist(g_means, bins=30, color='green', alpha=0.7, edgecolor='black')
axes[1].set_title('Green Channel')
axes[1].set_xlabel('Mean Pixel Value')
axes[1].axvline(np.mean(g_means), color='darkgreen', linestyle='--', label=f'Mean: {np.mean(g_means):.1f}')
axes[1].legend()

axes[2].hist(b_means, bins=30, color='blue', alpha=0.7, edgecolor='black')
axes[2].set_title('Blue Channel')
axes[2].set_xlabel('Mean Pixel Value')
axes[2].axvline(np.mean(b_means), color='darkblue', linestyle='--', label=f'Mean: {np.mean(b_means):.1f}')
axes[2].legend()

plt.tight_layout()
plt.savefig('/kaggle/working/color_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Color analysis done!")
print(f"   R Mean: {np.mean(r_means):.2f} | G Mean: {np.mean(g_means):.2f} | B Mean: {np.mean(b_means):.2f}")

In [ ]:

# ============================================================
# BRIGHTNESS ANALYSIS
# ============================================================

print("💡 Analyzing brightness...")
brightness_values = []

for path in color_sample:
    try:
        with Image.open(path).convert('L') as img:
            img_resized = img.resize((64, 64))
            brightness_values.append(np.array(img_resized).mean())
    except Exception:
        pass

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('💡 Brightness Analysis', fontsize=14, fontweight='bold')

axes[0].hist(brightness_values, bins=40, color='#F39C12', edgecolor='black', alpha=0.8)
axes[0].set_title('Brightness Distribution')
axes[0].set_xlabel('Mean Brightness (0-255)')
axes[0].set_ylabel('Count')
axes[0].axvline(np.mean(brightness_values), color='red', linestyle='--',
                label=f'Mean: {np.mean(brightness_values):.1f}')
axes[0].legend()

categories = ['Dark\n(0-85)', 'Medium\n(85-170)', 'Bright\n(170-255)']
counts = [
    sum(1 for b in brightness_values if b < 85),
    sum(1 for b in brightness_values if 85 <= b < 170),
    sum(1 for b in brightness_values if b >= 170)
]
colors = ['#2C3E50', '#E67E22', '#F1C40F']
axes[1].bar(categories, counts, color=colors, edgecolor='black')
axes[1].set_title('Brightness Categories')
axes[1].set_ylabel('Count')
for i, (cat, count) in enumerate(zip(categories, counts)):
    axes[1].text(i, count + 1, str(count), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('/kaggle/working/brightness_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ Brightness Analysis:")
print(f"   Mean: {np.mean(brightness_values):.2f}")
print(f"   Std:  {np.std(brightness_values):.2f}")
print(f"   Min:  {min(brightness_values):.2f}")
print(f"   Max:  {max(brightness_values):.2f}")

In [ ]:

# ============================================================
# FIND CORRUPTED IMAGES (Full Dataset Scan)
# ============================================================

print("🔍 Scanning ALL images for corruption...")
print("⏳ This may take 2-3 minutes...\n")

from tqdm.notebook import tqdm

corrupted_images = []
valid_images = []

for path in tqdm(all_image_paths, desc="Scanning"):
    try:
        with Image.open(path) as img:
            img.verify()
        valid_images.append(path)
    except Exception as e:
        corrupted_images.append((path, str(e)))

print(f"\n{'='*45}")
print(f"📊 CORRUPTION SCAN RESULTS")
print(f"{'='*45}")
print(f"✅ Valid Images    : {len(valid_images)}")
print(f"❌ Corrupted Images: {len(corrupted_images)}")
print(f"📊 Total Scanned   : {len(all_image_paths)}")
print(f"💯 Success Rate    : {len(valid_images)/len(all_image_paths)*100:.2f}%")
print(f"{'='*45}")

if corrupted_images:
    print("\n❌ Corrupted files:")
    for path, error in corrupted_images[:10]:
        print(f"   {os.path.basename(path)}: {error}")

In [ ]:

# ============================================================
# FINAL SUMMARY REPORT
# ============================================================

print("="*55)
print("🎌 DATA EXPLORATION - COMPLETE SUMMARY REPORT")
print("="*55)
print(f"📁 Dataset Path        : {DATASET_PATH}")
print(f"🖼️  Total Images        : {len(all_image_paths)}")
print(f"✅ Valid Images        : {len(valid_images)}")
print(f"❌ Corrupted Images    : {len(corrupted_images)}")
print(f"📐 Avg Width           : {np.mean(widths):.1f}px")
print(f"📐 Avg Height          : {np.mean(heights):.1f}px")
print(f"🔴 Avg Red Channel     : {np.mean(r_means):.2f}")
print(f"🟢 Avg Green Channel   : {np.mean(g_means):.2f}")
print(f"🔵 Avg Blue Channel    : {np.mean(b_means):.2f}")
print(f"💡 Avg Brightness      : {np.mean(brightness_values):.2f}")
print("="*55)
print("\n📁 FILES SAVED TO /kaggle/working/:")
print("   ✅ sample_grid.png")
print("   ✅ size_distribution.png")
print("   ✅ color_distribution.png")
print("   ✅ brightness_analysis.png")
print("\n🚀 NEXT STEP → Run 02_data_preprocessing.ipynb")
print("="*55)

# Save summary to CSV
summary_data = {
    'Metric': [
        'Total Images', 'Valid Images', 'Corrupted Images',
        'Avg Width', 'Avg Height', 'Avg Red', 'Avg Green',
        'Avg Blue', 'Avg Brightness'
    ],
    'Value': [
        len(all_image_paths), len(valid_images), len(corrupted_images),
        round(np.mean(widths), 2), round(np.mean(heights), 2),
        round(np.mean(r_means), 2), round(np.mean(g_means), 2),
        round(np.mean(b_means), 2), round(np.mean(brightness_values), 2)
    ]
}

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv('/kaggle/working/exploration_summary.csv', index=False)
print("\n✅ Summary saved to exploration_summary.csv")
print("🎉 Notebook 01 - DATA EXPLORATION COMPLETE!")

In [ ]:

# ============================================================
# 🎌 NOTEBOOK 02 - DATA PREPROCESSING STARTS HERE
# ============================================================
print("=" * 55)
print("📓 NOTEBOOK 02 - DATA PREPROCESSING")
print("=" * 55)

In [ ]:

import os
import json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError
from tqdm.notebook import tqdm
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.utils as vutils
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported!")
print(f"🔥 PyTorch Version : {torch.__version__}")
print(f"💻 CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🚀 GPU             : {torch.cuda.get_device_name(0)}")

In [ ]:

# ============================================================
# CONFIGURATION - USING CORRECT PATH FROM NOTEBOOK 01
# ============================================================

# ✅ Correct path already found in Notebook 01
DATASET_PATH = '/kaggle/input/datasets/splcher/animefacedataset/images'
OUTPUT_PATH  = '/kaggle/working'
IMAGE_SIZE   = 64
CHANNELS     = 3
BATCH_SIZE   = 128
NUM_WORKERS  = 2
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(f'{OUTPUT_PATH}/plots', exist_ok=True)

print("="*45)
print("⚙️  CONFIGURATION")
print("="*45)
print(f"📁 Dataset Path  : {DATASET_PATH}")
print(f"📁 Path Exists   : {os.path.exists(DATASET_PATH)}")
print(f"📁 Output Path   : {OUTPUT_PATH}")
print(f"📐 Image Size    : {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"📦 Batch Size    : {BATCH_SIZE}")
print(f"👷 Num Workers   : {NUM_WORKERS}")
print(f"💻 Device        : {DEVICE}")
if torch.cuda.is_available():
    print(f"🚀 GPU           : {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("="*45)

In [ ]:

# ============================================================
# COLLECT AND VALIDATE ALL IMAGES
# ============================================================

image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
all_paths        = []

for root, dirs, files in os.walk(DATASET_PATH):
    for file in files:
        if os.path.splitext(file)[1].lower() in image_extensions:
            all_paths.append(os.path.join(root, file))

print(f"📊 Total images found: {len(all_paths)}")
print("🔍 Validating images...")

valid_paths   = []
corrupt_paths = []

for path in tqdm(all_paths, desc="Validating"):
    try:
        with Image.open(path) as img:
            img.verify()
        valid_paths.append(path)
    except Exception:
        corrupt_paths.append(path)

print(f"\n{'='*40}")
print(f"✅ Valid Images    : {len(valid_paths)}")
print(f"❌ Corrupt Images  : {len(corrupt_paths)}")
print(f"💯 Valid Rate      : {len(valid_paths)/len(all_paths)*100:.2f}%")
print(f"{'='*40}")

In [ ]:

# ============================================================
# DEFINE TRANSFORMS
# ============================================================

train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1,
        saturation=0.1,
        hue=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

inference_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

print("✅ Transforms Defined!")
print("\n🔄 TRAINING TRANSFORMS:")
print("   1. Resize         → 64x64")
print("   2. H-Flip         → p=0.5")
print("   3. Color Jitter   → brightness/contrast/saturation/hue")
print("   4. ToTensor       → [0,1]")
print("   5. Normalize      → [-1, 1]")
print("\n🔄 INFERENCE TRANSFORMS:")
print("   1. Resize         → 64x64")
print("   2. ToTensor       → [0,1]")
print("   3. Normalize      → [-1, 1]")

In [ ]:

# ============================================================
# CUSTOM DATASET CLASS
# ============================================================

class AnimeFaceDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform   = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        try:
            image = Image.open(path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image
        except Exception:
            return torch.zeros(3, IMAGE_SIZE, IMAGE_SIZE)

# Create dataset
dataset = AnimeFaceDataset(
    image_paths=valid_paths,
    transform=train_transforms
)

sample_image = dataset[0]
print(f"✅ Dataset Created!")
print(f"📊 Total Samples  : {len(dataset)}")
print(f"🖼️  Image Shape    : {sample_image.shape}")
print(f"📉 Pixel Min      : {sample_image.min():.4f}")
print(f"📈 Pixel Max      : {sample_image.max():.4f}")
print(f"✅ Normalized     : {-1.1 < sample_image.min() and sample_image.max() < 1.1}")

In [ ]:

# ============================================================
# CREATE DATALOADER
# ============================================================

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True if torch.cuda.is_available() else False,
    drop_last=True
)

sample_batch = next(iter(dataloader))

print(f"✅ DataLoader Created!")
print(f"{'='*40}")
print(f"📦 Batch Size     : {BATCH_SIZE}")
print(f"🔢 Total Batches  : {len(dataloader)}")
print(f"👷 Num Workers    : {NUM_WORKERS}")
print(f"📌 Pin Memory     : {torch.cuda.is_available()}")
print(f"{'='*40}")
print(f"\n✅ Test Batch:")
print(f"   Shape           : {sample_batch.shape}")
print(f"   Min             : {sample_batch.min():.4f}")
print(f"   Max             : {sample_batch.max():.4f}")

In [ ]:

# ============================================================
# VISUALIZE PREPROCESSED IMAGES
# ============================================================

def denormalize(tensor):
    """Convert from [-1,1] back to [0,1] for display"""
    return (tensor + 1) / 2

sample_batch = next(iter(dataloader))

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
fig.suptitle(
    '✅ Preprocessed Images (64x64, Normalized to [-1,1])',
    fontsize=14, fontweight='bold'
)

for idx, ax in enumerate(axes.flatten()):
    img = denormalize(sample_batch[idx])
    img = img.permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)
    ax.imshow(img)
    ax.axis('off')

plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/plots/preprocessed_samples.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Preprocessed samples saved!")

In [ ]:

# ============================================================
# AUGMENTATION COMPARISON
# ============================================================

import random
sample_path  = random.choice(valid_paths)
original_img = Image.open(sample_path).convert('RGB')

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle(
    '🔄 Augmentation Examples (Same Image, Different Results)',
    fontsize=14, fontweight='bold'
)

axes[0][0].imshow(original_img.resize((64, 64)))
axes[0][0].set_title('Original', fontweight='bold', color='green')
axes[0][0].axis('off')

for i in range(1, 10):
    row = i // 5
    col = i % 5
    aug_tensor = train_transforms(original_img)
    aug_img    = denormalize(aug_tensor).permute(1, 2, 0).numpy()
    aug_img    = np.clip(aug_img, 0, 1)
    axes[row][col].imshow(aug_img)
    axes[row][col].set_title(f'Augmented #{i}', fontsize=9)
    axes[row][col].axis('off')

plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/plots/augmentation_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Augmentation comparison saved!")

In [ ]:

# ============================================================
# FINAL VERIFICATION & SAVE DATASET INFO
# ============================================================

print("🔍 Running final verification...")
min_val = float('inf')
max_val = float('-inf')

for i, batch in enumerate(tqdm(dataloader, desc="Verifying")):
    min_val = min(min_val, batch.min().item())
    max_val = max(max_val, batch.max().item())
    if i >= 20:
        break

# Save dataset info for Notebook 03
dataset_info = {
    'dataset_path'   : DATASET_PATH,
    'total_images'   : len(all_paths),
    'valid_images'   : len(valid_paths),
    'corrupt_images' : len(corrupt_paths),
    'image_size'     : IMAGE_SIZE,
    'channels'       : CHANNELS,
    'batch_size'     : BATCH_SIZE,
    'total_batches'  : len(dataloader),
    'pixel_min'      : round(min_val, 4),
    'pixel_max'      : round(max_val, 4),
    'normalize_mean' : [0.5, 0.5, 0.5],
    'normalize_std'  : [0.5, 0.5, 0.5],
    'device'         : str(DEVICE)
}

with open(f'{OUTPUT_PATH}/dataset_info.json', 'w') as f:
    json.dump(dataset_info, f, indent=4)

print(f"\n{'='*55}")
print(f"🎌 NOTEBOOK 02 - COMPLETE SUMMARY")
print(f"{'='*55}")
print(f"✅ Total Images       : {len(all_paths)}")
print(f"✅ Valid Images       : {len(valid_paths)}")
print(f"❌ Corrupt Images     : {len(corrupt_paths)}")
print(f"📐 Image Size         : {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"📦 Batch Size         : {BATCH_SIZE}")
print(f"🔢 Total Batches      : {len(dataloader)}")
print(f"📉 Pixel Min          : {min_val:.4f} (expected ~-1.0)")
print(f"📈 Pixel Max          : {max_val:.4f} (expected ~+1.0)")
print(f"💻 Device             : {DEVICE}")
print(f"{'='*55}")
print(f"\n📁 FILES SAVED TO /kaggle/working/plots/:")
print(f"   ✅ preprocessed_samples.png")
print(f"   ✅ augmentation_comparison.png")
print(f"\n📁 FILES SAVED TO /kaggle/working/:")
print(f"   ✅ dataset_info.json")
print(f"\n🚀 NEXT STEP → Add Notebook 03 cells below!")
print(f"{'='*55}")
print(f"🎉 NOTEBOOK 02 - DATA PREPROCESSING COMPLETE!")

**📓 NOTEBOOK 03 - MODEL ARCHITECTURE (DCGAN)**


In [ ]:

# ============================================================
# 🎌 NOTEBOOK 03 - MODEL ARCHITECTURE START
# DCGAN: Generator & Discriminator definitions
# ============================================================

import os
import math
import torch
import torch.nn as nn

# Load image/dataloader info
OUTPUT_PATH = '/kaggle/working'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

dataset_info_path = os.path.join(OUTPUT_PATH, 'dataset_info.json')
assert os.path.exists(dataset_info_path), f"dataset_info.json not found at {dataset_info_path}"

import json
with open(dataset_info_path, 'r') as f:
    dataset_info = json.load(f)

IMAGE_SIZE  = dataset_info.get('image_size', 64)
CHANNELS    = dataset_info.get('channels', 3)

print("="*55)
print("🎌 NOTEBOOK 03 - LOADED DATASET INFO")
print("="*55)
print(dataset_info)
print("="*55)
print(f"💻 Device: {DEVICE}")

In [ ]:

# ============================================================
# DCGAN HYPERPARAMETERS (as per your project plan)
# ============================================================

LATENT_DIM = 100
NGF = 512  # generator feature base
NDF = 512  # discriminator feature base

# Output channels = 3 (RGB), input channels = 3
img_size = IMAGE_SIZE

# Helper: ensure img_size is power of 2 for DCGAN
assert img_size in [64], f"Expected img_size=64, got {img_size}"

print("✅ DCGAN parameters set:")
print(f"latent_dim = {LATENT_DIM}")
print(f"img_size    = {img_size}x{img_size}")
print(f"channels    = {CHANNELS}")

In [ ]:

# ============================================================
# WEIGHT INITIALIZATION
# ============================================================

def weights_init_dcgan(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

print("✅ weights_init_dcgan ready")

In [ ]:

# ============================================================
# GENERATOR (DCGAN)
# Input: z (N, 100, 1, 1)
# Output: image (N, 3, 64, 64) in [-1, 1] due to Tanh
# ============================================================

class Generator(nn.Module):
    def __init__(self, latent_dim=100, img_channels=3, feature_maps=512):
        super().__init__()

        # 1x1 -> 4x4 -> 8x8 -> 16x16 -> 32x32 -> 64x64
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, feature_maps, kernel_size=4, stride=1, padding=0, bias=False),  # 1->4
            nn.BatchNorm2d(feature_maps),
            nn.ReLU(True),

            nn.ConvTranspose2d(feature_maps, feature_maps // 2, kernel_size=4, stride=2, padding=1, bias=False),  # 4->8
            nn.BatchNorm2d(feature_maps // 2),
            nn.ReLU(True),

            nn.ConvTranspose2d(feature_maps // 2, feature_maps // 4, kernel_size=4, stride=2, padding=1, bias=False),  # 8->16
            nn.BatchNorm2d(feature_maps // 4),
            nn.ReLU(True),

            nn.ConvTranspose2d(feature_maps // 4, feature_maps // 8, kernel_size=4, stride=2, padding=1, bias=False),  # 16->32
            nn.BatchNorm2d(feature_maps // 8),
            nn.ReLU(True),

            nn.ConvTranspose2d(feature_maps // 8, img_channels, kernel_size=4, stride=2, padding=1, bias=False),  # 32->64
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)

In [ ]:

# ============================================================
# DISCRIMINATOR (DCGAN)
# Input: image (N, 3, 64, 64)
# Output: probability (N, 1) via Sigmoid
# ============================================================

class Discriminator(nn.Module):
    def __init__(self, img_channels=3, feature_maps=512):
        super().__init__()

        # 64->32->16->8->4->1
        self.net = nn.Sequential(
            nn.Conv2d(img_channels, feature_maps // 8, kernel_size=4, stride=2, padding=1, bias=False),  # 64->32
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(feature_maps // 8, feature_maps // 4, kernel_size=4, stride=2, padding=1, bias=False),  # 32->16
            nn.BatchNorm2d(feature_maps // 4),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(feature_maps // 4, feature_maps // 2, kernel_size=4, stride=2, padding=1, bias=False),  # 16->8
            nn.BatchNorm2d(feature_maps // 2),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(feature_maps // 2, feature_maps, kernel_size=4, stride=2, padding=1, bias=False),  # 8->4
            nn.BatchNorm2d(feature_maps),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(feature_maps, 1, kernel_size=4, stride=1, padding=0, bias=False),  # 4->1
            nn.Sigmoid()
        )

    def forward(self, x):
        # output shape: (N, 1, 1, 1) -> (N, 1)
        out = self.net(x)
        return out.view(out.size(0), -1)

In [ ]:

# ============================================================
# INSTANTIATE MODELS + APPLY WEIGHTS INIT
# TEST FORWARD PASS SHAPES
# ============================================================

G = Generator(latent_dim=LATENT_DIM, img_channels=CHANNELS, feature_maps=NGF).to(DEVICE)
D = Discriminator(img_channels=CHANNELS, feature_maps=NDF).to(DEVICE)

G.apply(weights_init_dcgan)
D.apply(weights_init_dcgan)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("="*55)
print("🎛️ MODEL SUMMARY (PARAMETERS)")
print("="*55)
print(f"Generator trainable params    : {count_parameters(G):,}")
print(f"Discriminator trainable params: {count_parameters(D):,}")
print("="*55)

# Test forward pass
batch_size = 16
z = torch.randn(batch_size, LATENT_DIM, 1, 1, device=DEVICE)
fake_images = G(z)
print("✅ Generator output shape:", fake_images.shape)  # (N, 3, 64, 64)

dummy_real = torch.randn(batch_size, CHANNELS, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
real_probs = D(dummy_real)
fake_probs = D(fake_images.detach())

print("✅ Discriminator output shape (real):", real_probs.shape)  # (N, 1)
print("✅ Discriminator output shape (fake):", fake_probs.shape)  # (N, 1)
print("✅ D(real) range:", (real_probs.min().item(), real_probs.max().item()))
print("✅ D(fake) range:", (fake_probs.min().item(), fake_probs.max().item()))
print("="*55)

** Notebook 03 - Save Architectures (optional) and Next Step**

In [ ]:

# ============================================================
# LOSS SETUP (for later training notebook)
# ============================================================

import torch.nn.functional as F

criterion = torch.nn.BCELoss()

print("="*55)
print("🧾 Training components prepared for Notebook 04")
print("="*55)
print("Loss Function (GAN): BCEWithLogits? -> using BCELoss with Sigmoid output in D")
print(f"criterion: {criterion.__class__.__name__}")

# Save lightweight architecture checkpoints (optional)
# (Only saving state dicts of initialized models; actual training will overwrite these.)
ckpt_dir = os.path.join(OUTPUT_PATH, "checkpoints")
os.makedirs(ckpt_dir, exist_ok=True)

torch.save(G.state_dict(), os.path.join(ckpt_dir, "generator_init.pth"))
torch.save(D.state_dict(), os.path.join(ckpt_dir, "discriminator_init.pth"))

print(f"\n✅ Saved initialized weights to: {ckpt_dir}")
print("   generator_init.pth")
print("   discriminator_init.pth")

print("\n🚀 NEXT STEP → Add Notebook 04 cells below: TRAINING")
print("="*55)

In [ ]:

# ============================================================
# CHECKPOINTING + RESUME UTILITIES (for Notebook 04)
# ============================================================

import os
import torch

ckpt_dir = os.path.join(OUTPUT_PATH, "checkpoints")
os.makedirs(ckpt_dir, exist_ok=True)

# We'll save:
# - generator_epoch_{epoch}.pth
# - discriminator_epoch_{epoch}.pth
# - optimG_epoch_{epoch}.pth
# - optimD_epoch_{epoch}.pth
# - training_state.pth (latest resume info)
#
# Resume uses training_state.pth
training_state_path = os.path.join(ckpt_dir, "training_state.pth")

def save_checkpoint(epoch, G, D, optimG, optimD, g_loss, d_loss, fixed_noise, fid_history=None):
    state = {
        "epoch": epoch,
        "generator_state_dict": G.state_dict(),
        "discriminator_state_dict": D.state_dict(),
        "optimG_state_dict": optimG.state_dict(),
        "optimD_state_dict": optimD.state_dict(),
        "g_loss": float(g_loss),
        "d_loss": float(d_loss),
        "fixed_noise_shape": tuple(fixed_noise.shape),
        "fid_history": fid_history if fid_history is not None else None,
    }
    # Latest resume file
    torch.save(state, training_state_path)

    # Also save per-epoch files (extra safety)
    torch.save(G.state_dict(), os.path.join(ckpt_dir, f"generator_epoch_{epoch}.pth"))
    torch.save(D.state_dict(), os.path.join(ckpt_dir, f"discriminator_epoch_{epoch}.pth"))
    torch.save(optimG.state_dict(), os.path.join(ckpt_dir, f"optimG_epoch_{epoch}.pth"))
    torch.save(optimD.state_dict(), os.path.join(ckpt_dir, f"optimD_epoch_{epoch}.pth"))

    print(f"✅ Checkpoint saved for epoch {epoch} at: {training_state_path}")

def load_checkpoint_if_exists(G, D, optimG, optimD, map_location=None):
    if not os.path.exists(training_state_path):
        print("ℹ️ No checkpoint found. Starting fresh.")
        return 0, None

    state = torch.load(training_state_path, map_location=map_location)
    G.load_state_dict(state["generator_state_dict"])
    D.load_state_dict(state["discriminator_state_dict"])
    optimG.load_state_dict(state["optimG_state_dict"])
    optimD.load_state_dict(state["optimD_state_dict"])

    start_epoch = int(state["epoch"]) + 1  # resume AFTER last completed epoch
    print(f"✅ Resumed from checkpoint. Last epoch: {state['epoch']} → starting at epoch {start_epoch}")

    return start_epoch, state

In [ ]:

# ============================================================
# FIXED NOISE (for consistent visuals per epoch)
# + Attempt resume
# ============================================================

LATENT_DIM = 100  # keep consistent with Notebook 03

fixed_noise = torch.randn(64, LATENT_DIM, 1, 1, device=DEVICE)

print(f"✅ fixed_noise shape: {fixed_noise.shape}")

# In Notebook 04, after you create G, D, optimG, optimD, call:
# start_epoch, last_state = load_checkpoint_if_exists(G, D, optimG, optimD, map_location=DEVICE)

In [ ]:

# ============================================================
# OPTIMIZERS (use same hyperparams as project plan)
# ============================================================

LR = 0.0002
beta1 = 0.5
beta2 = 0.999

criterion = torch.nn.BCELoss()  # Discriminator uses Sigmoid

# Create optimizers AFTER G and D are defined
optimG = torch.optim.Adam(G.parameters(), lr=LR, betas=(beta1, beta2))
optimD = torch.optim.Adam(D.parameters(), lr=LR, betas=(beta1, beta2))

print("✅ Optimizers created")

In [ ]:


print(f"🚀 Starting training from epoch: {start_epoch}")

In [ ]:

import torch
state = torch.load("/kaggle/working/checkpoints/training_state.pth", map_location="cpu")
print("Last epoch stored now:", state["epoch"])

In [ ]:

print("start_epoch from checkpoint will be:", start_epoch)
print("EPOCHS is:", EPOCHS)
print("will run epochs:", list(range(start_epoch, EPOCHS))[:10], "...", "count:", len(list(range(start_epoch, EPOCHS))))

In [ ]:

import os, torch

print("generator_epoch_50 exists?:", os.path.exists("/kaggle/working/checkpoints/generator_epoch_50.pth"))

state = torch.load("/kaggle/working/checkpoints/training_state.pth", map_location="cpu")
print("Last epoch stored now:", state["epoch"])

In [ ]:

# ============================================================
# NOTEBOOK 04 - TRAINING (DCGAN) WITH CHECKPOINT RESUME
# ============================================================

import os
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision.utils as vutils
import matplotlib.pyplot as plt

# ----------------------------
# Basic settings
# ----------------------------
EPOCHS = 50
LR = 0.0002
beta1 = 0.5
beta2 = 0.999

# Your current Discriminator ends with nn.Sigmoid(),
# so D outputs probabilities in [0, 1].
#
# Perfect loss choice:
# - Discriminator output: probability (after Sigmoid)
# - Loss: BCELoss expects probabilities (not logits)
criterion = nn.BCELoss()

# Optimizers
optimG = torch.optim.Adam(G.parameters(), lr=LR, betas=(beta1, beta2))
optimD = torch.optim.Adam(D.parameters(), lr=LR, betas=(beta1, beta2))

# Move criterion to device (safe, though BCELoss usually fine on CPU)
criterion = criterion.to(DEVICE)

# Checkpoint utilities
ckpt_dir = os.path.join(OUTPUT_PATH, "checkpoints")
os.makedirs(ckpt_dir, exist_ok=True)
training_state_path = os.path.join(ckpt_dir, "training_state.pth")

def save_checkpoint(epoch, G, D, optimG, optimD, g_loss, d_loss):
    state = {
        "epoch": epoch,
        "generator_state_dict": G.state_dict(),
        "discriminator_state_dict": D.state_dict(),
        "optimG_state_dict": optimG.state_dict(),
        "optimD_state_dict": optimD.state_dict(),
        "g_loss": float(g_loss),
        "d_loss": float(d_loss),
    }
    torch.save(state, training_state_path)

    # Extra per-epoch files (helps if you want manual rollback)
    torch.save(G.state_dict(), os.path.join(ckpt_dir, f"generator_epoch_{epoch}.pth"))
    torch.save(D.state_dict(), os.path.join(ckpt_dir, f"discriminator_epoch_{epoch}.pth"))
    torch.save(optimG.state_dict(), os.path.join(ckpt_dir, f"optimG_epoch_{epoch}.pth"))
    torch.save(optimD.state_dict(), os.path.join(ckpt_dir, f"optimD_epoch_{epoch}.pth"))

    print(f"✅ Checkpoint saved: epoch {epoch}")

def load_checkpoint_if_exists(G, D, optimG, optimD):
    if not os.path.exists(training_state_path):
        print("ℹ️ No checkpoint found. Starting fresh.")
        return 0

    state = torch.load(training_state_path, map_location=DEVICE)
    G.load_state_dict(state["generator_state_dict"])
    D.load_state_dict(state["discriminator_state_dict"])
    optimG.load_state_dict(state["optimG_state_dict"])
    optimD.load_state_dict(state["optimD_state_dict"])

    # Resume from next epoch (since current epoch already completed)
    start_epoch = int(state["epoch"]) + 1
    print(f"✅ Resumed from checkpoint. Last completed epoch: {state['epoch']} → start at {start_epoch}")
    return start_epoch

# ----------------------------
# Resume if checkpoint exists
# ----------------------------
start_epoch = load_checkpoint_if_exists(G, D, optimG, optimD)

# ----------------------------
# Fixed noise for visualization
# ----------------------------
# You already created fixed_noise earlier, but if not, create it here:
if "fixed_noise" not in globals():
    fixed_noise = torch.randn(64, 100, 1, 1, device=DEVICE)

# ----------------------------
# DataLoader
# ----------------------------
# IMPORTANT: Notebook 02 created `dataloader`.
# If you're continuing in the same notebook and dataloader exists, we use it.
assert "dataloader" in globals(), "dataloader not found. Run Notebook 02 cells or create dataloader."

# ----------------------------
# Training loop
# ----------------------------
G.to(DEVICE).train()
D.to(DEVICE).train()

loss_history_path = os.path.join(OUTPUT_PATH, "loss_history.csv")
loss_rows = []

# If resuming, optionally load existing loss_history.csv
if os.path.exists(loss_history_path):
    old_df = pd.read_csv(loss_history_path)
    loss_rows = old_df.to_dict(orient="records")
    print(f"ℹ️ Loaded existing loss history with {len(loss_rows)} rows.")
else:
    print("ℹ️ No existing loss_history.csv found. Will create a new one.")

# Directory for generated samples during training
gen_dir = os.path.join(OUTPUT_PATH, "generated_images")
os.makedirs(gen_dir, exist_ok=True)

def denorm(x):
    # x in [-1, 1] -> [0, 1]
    return (x + 1) / 2

for epoch in range(start_epoch, EPOCHS):
    t0 = time.time()
    g_running = 0.0
    d_running = 0.0

    for i, real_imgs in enumerate(dataloader):
        real_imgs = real_imgs.to(DEVICE)

        # Label shapes:
        # Discriminator outputs (N, 1), since it does view(out.size(0), -1)
        # So labels must be (N, 1) for BCELoss.
        batch_size = real_imgs.size(0)
        real_labels = torch.ones(batch_size, 1, device=DEVICE)
        fake_labels = torch.zeros(batch_size, 1, device=DEVICE)

        # ----------------------------
        # 1) Train Discriminator
        # ----------------------------
        D.zero_grad(set_to_none=True)

        # Real loss
        real_preds = D(real_imgs)  # (N, 1)
        d_real_loss = criterion(real_preds, real_labels)

        # Fake loss
        z = torch.randn(batch_size, 100, 1, 1, device=DEVICE)
        fake_imgs = G(z)

        fake_preds = D(fake_imgs.detach())  # detach so G not updated here
        d_fake_loss = criterion(fake_preds, fake_labels)

        d_loss = d_real_loss + d_fake_loss
        d_loss.backward()
        optimD.step()

        # ----------------------------
        # 2) Train Generator
        # ----------------------------
        G.zero_grad(set_to_none=True)

        # Generator wants discriminator to classify fakes as real => labels = 1
        z = torch.randn(batch_size, 100, 1, 1, device=DEVICE)
        gen_imgs = G(z)
        gen_preds = D(gen_imgs)  # (N, 1)
        g_loss = criterion(gen_preds, real_labels)

        g_loss.backward()
        optimG.step()

        g_running += g_loss.item()
        d_running += d_loss.item()

        if (i + 1) % 20 == 0:
            print(f"Epoch [{epoch+1}/{EPOCHS}] Step [{i+1}/{len(dataloader)}] "
                  f"D_loss: {d_loss.item():.4f} G_loss: {g_loss.item():.4f}")

    # Average losses for epoch
    avg_g = g_running / len(dataloader)
    avg_d = d_running / len(dataloader)

    epoch_time = time.time() - t0
    print(f"\n✅ Epoch {epoch+1}/{EPOCHS} done in {epoch_time:.1f}s | "
          f"Avg D_loss={avg_d:.4f}, Avg G_loss={avg_g:.4f}")

    # ----------------------------
    # Save generated samples (fixed noise)
    # ----------------------------
    G.eval()
    with torch.no_grad():
        fake_fixed = G(fixed_noise).detach()  # (64,3,64,64)
        fake_fixed = denorm(fake_fixed)
        grid = vutils.make_grid(fake_fixed, nrow=8, padding=2, normalize=False)

        # Save image
        out_path = os.path.join(gen_dir, f"epoch_{epoch+1:03d}_grid.png")
        plt.figure(figsize=(10, 10))
        plt.axis("off")
        plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
        plt.tight_layout()
        plt.savefig(out_path, dpi=150, bbox_inches="tight")
        plt.close()

    G.train()

    # ----------------------------
    # Save checkpoint EVERY epoch (resume-friendly)
    # ----------------------------
    save_checkpoint(epoch, G, D, optimG, optimD, g_loss=avg_g, d_loss=avg_d)

    # ----------------------------
    # Save loss history to CSV EVERY epoch
    # ----------------------------
    loss_rows.append({
        "epoch": epoch + 1,
        "g_loss": avg_g,
        "d_loss": avg_d
    })
    pd.DataFrame(loss_rows).to_csv(loss_history_path, index=False)

    print(f"📈 Updated loss_history.csv (latest epoch {epoch+1})")
    print("-" * 70)

training_state.pth exists: True

In [ ]:

import os, time
import torch
import torch.nn as nn
import pandas as pd
import torchvision.utils as vutils
import matplotlib.pyplot as plt

# Ensure we train exactly epoch 50
TARGET_EPOCH = 50

# Loss for Sigmoid-based D
criterion = nn.BCELoss()

# Checkpoint paths
ckpt_dir = os.path.join(OUTPUT_PATH, "checkpoints")
os.makedirs(ckpt_dir, exist_ok=True)
training_state_path = os.path.join(ckpt_dir, "training_state.pth")

# Load latest training_state for optimizer/model states
optimG = torch.optim.Adam(G.parameters(), lr=LR, betas=(beta1, beta2))
optimD = torch.optim.Adam(D.parameters(), lr=LR, betas=(beta1, beta2))

start_epoch = 0
if os.path.exists(training_state_path):
    state = torch.load(training_state_path, map_location=DEVICE)
    start_epoch = int(state["epoch"]) + 1  # next epoch index your code uses
    G.load_state_dict(state["generator_state_dict"])
    D.load_state_dict(state["discriminator_state_dict"])
    optimG.load_state_dict(state["optimG_state_dict"])
    optimD.load_state_dict(state["optimD_state_dict"])
    print(f"✅ Loaded checkpoint. last_epoch={state['epoch']} start_epoch={start_epoch}")
else:
    print("ℹ️ No training_state found.")

print("Will run epoch index:", TARGET_EPOCH)
assert TARGET_EPOCH >= start_epoch, "Checkpoint is ahead of target epoch; cannot go back."

In [ ]:

import torch
from torch.nn import functional as F

def denorm(x):
    return (x + 1) / 2

# Ensure fixed_noise, dataloader exist
if "fixed_noise" not in globals():
    fixed_noise = torch.randn(64, 100, 1, 1, device=DEVICE)

assert "dataloader" in globals(), "dataloader not found."
assert "denorm" in globals(), "denorm not found."

# Label tensors helpers (shape must match D output: (N,1))
def make_labels(batch_size, value):
    return torch.full((batch_size, 1), float(value), device=DEVICE)

criterion = criterion.to(DEVICE)
G.to(DEVICE).train()
D.to(DEVICE).train()

# Train exactly one epoch: TARGET_EPOCH
epoch = TARGET_EPOCH
t0 = time.time()
g_running = 0.0
d_running = 0.0

for i, real_imgs in enumerate(dataloader):
    real_imgs = real_imgs.to(DEVICE)
    batch_size = real_imgs.size(0)

    real_labels = make_labels(batch_size, 1)
    fake_labels = make_labels(batch_size, 0)

    # ---- Train D ----
    D.zero_grad(set_to_none=True)
    real_preds = D(real_imgs)  # (N,1) after Sigmoid
    d_real_loss = criterion(real_preds, real_labels)

    z = torch.randn(batch_size, 100, 1, 1, device=DEVICE)
    fake_imgs = G(z)

    fake_preds = D(fake_imgs.detach())
    d_fake_loss = criterion(fake_preds, fake_labels)

    d_loss = d_real_loss + d_fake_loss
    d_loss.backward()
    optimD.step()

    # ---- Train G ----
    G.zero_grad(set_to_none=True)
    z = torch.randn(batch_size, 100, 1, 1, device=DEVICE)
    gen_imgs = G(z)
    gen_preds = D(gen_imgs)
    g_loss = criterion(gen_preds, real_labels)

    g_loss.backward()
    optimG.step()

    g_running += g_loss.item()
    d_running += d_loss.item()

    if (i + 1) % 20 == 0:
        print(f"Epoch {epoch} Step [{i+1}/{len(dataloader)}] D_loss={d_loss.item():.4f} G_loss={g_loss.item():.4f}")

avg_g = g_running / len(dataloader)
avg_d = d_running / len(dataloader)

print(f"\n✅ Completed training for epoch {epoch} in {time.time()-t0:.1f}s | AvgD={avg_d:.4f} AvgG={avg_g:.4f}")

# ---- Force save checkpoints for epoch 50 ----
state = {
    "epoch": epoch,
    "generator_state_dict": G.state_dict(),
    "discriminator_state_dict": D.state_dict(),
    "optimG_state_dict": optimG.state_dict(),
    "optimD_state_dict": optimD.state_dict(),
    "g_loss": float(avg_g),
    "d_loss": float(avg_d),
}
torch.save(state, training_state_path)

torch.save(G.state_dict(), os.path.join(ckpt_dir, f"generator_epoch_{epoch}.pth"))
torch.save(D.state_dict(), os.path.join(ckpt_dir, f"discriminator_epoch_{epoch}.pth"))
torch.save(optimG.state_dict(), os.path.join(ckpt_dir, f"optimG_epoch_{epoch}.pth"))
torch.save(optimD.state_dict(), os.path.join(ckpt_dir, f"optimD_epoch_{epoch}.pth"))

print("✅ Forced checkpoint saved for epoch 50.")

In [ ]:

import os
ckpt_dir = "/kaggle/working/checkpoints"
print("generator_epoch_50 exists?:", os.path.exists(os.path.join(ckpt_dir, "generator_epoch_50.pth")))
print("discriminator_epoch_50 exists?:", os.path.exists(os.path.join(ckpt_dir, "discriminator_epoch_50.pth")))

import torch
state = torch.load(os.path.join(ckpt_dir, "training_state.pth"), map_location="cpu")
print("Last epoch stored now:", state["epoch"])

In [ ]:

import os, torch
ckpt_path = "/kaggle/working/checkpoints/training_state.pth"
print("Exists:", os.path.exists(ckpt_path))
if os.path.exists(ckpt_path):
    state = torch.load(ckpt_path, map_location="cpu")
    print("Last completed epoch saved in checkpoint:", state["epoch"])

In [ ]:

import os
ckpt_dir = "/kaggle/working/checkpoints"
print("Files:", sorted(os.listdir(ckpt_dir))[:20])
print("Has generator_epoch_36?:", os.path.exists(os.path.join(ckpt_dir, "generator_epoch_36.pth")))
print("Has generator_epoch_35?:", os.path.exists(os.path.join(ckpt_dir, "generator_epoch_35.pth")))
print("Has generator_epoch_49?:", os.path.exists(os.path.join(ckpt_dir, "generator_epoch_49.pth")))
print("Has generator_epoch_50?:", os.path.exists(os.path.join(ckpt_dir, "generator_epoch_50.pth")))

In [ ]:

import os, glob
paths = sorted(glob.glob("/kaggle/working/generated_images/epoch_0*_grid.png"))
paths[-5:], len(paths)

In [ ]:

import os

ckpt_dir = "/kaggle/working/checkpoints"
gen_files  = sorted([f for f in os.listdir(ckpt_dir) if f.startswith("generator_epoch_") and f.endswith(".pth")])
disc_files = sorted([f for f in os.listdir(ckpt_dir) if f.startswith("discriminator_epoch_") and f.endswith(".pth")])

def extract_epoch(fn, prefix):
    # example: generator_epoch_36.pth
    return int(fn.replace(prefix, "").replace(".pth",""))

gen_epochs  = sorted({extract_epoch(f, "generator_epoch_") for f in gen_files})
disc_epochs = sorted({extract_epoch(f, "discriminator_epoch_") for f in disc_files})

print("✅ Generator epochs present :", (gen_epochs[0], gen_epochs[-1], len(gen_epochs)))
print("✅ Discriminator epochs present:", (disc_epochs[0], disc_epochs[-1], len(disc_epochs)))

missing_gen  = [e for e in range(1, 51) if e not in set(gen_epochs)]
missing_disc = [e for e in range(1, 51) if e not in set(disc_epochs)]

print("\n🔴 Missing generator checkpoints:", missing_gen if missing_gen else "None")
print("🔴 Missing discriminator checkpoints:", missing_disc if missing_disc else "None")

In [ ]:

import glob

grid_dir = "/kaggle/working/generated_images"
grid_paths = sorted(glob.glob(os.path.join(grid_dir, "epoch_*_grid.png")))

def extract_epoch_grid(p):
    # /.../epoch_050_grid.png -> 50
    base = os.path.basename(p)
    # epoch_050_grid.png
    return int(base.split("_")[1])

grid_epochs = sorted({extract_epoch_grid(p) for p in grid_paths})

print("✅ Grid epochs present :", (grid_epochs[0], grid_epochs[-1], len(grid_epochs)))

missing_grids = [e for e in range(1, 51) if e not in set(grid_epochs)]
print("\n🔴 Missing grid images:", missing_grids if missing_grids else "None")

In [ ]:

import os
resume_path = "/kaggle/working/checkpoints/training_state.pth"
print("training_state.pth exists:", os.path.exists(resume_path))

In [ ]:

import os, glob, torch

ckpt_dir = "/kaggle/working/checkpoints"
gen_dir  = "/kaggle/working/generated_images"
state_path = os.path.join(ckpt_dir, "training_state.pth")

def list_epochs(prefix):
    files = glob.glob(os.path.join(ckpt_dir, f"{prefix}_epoch_*.pth"))
    epochs = []
    for f in files:
        base = os.path.basename(f)  # generator_epoch_36.pth
        ep = int(base.split("_epoch_")[1].split(".pth")[0])
        epochs.append(ep)
    return sorted(set(epochs)), sorted(files)

gen_epochs, gen_files = list_epochs("generator")
disc_epochs, disc_files = list_epochs("discriminator")

# Determine training_state epoch
state_epoch = None
if os.path.exists(state_path):
    state = torch.load(state_path, map_location="cpu")
    state_epoch = state.get("epoch", None)

# Grids: epoch_001_grid.png ... epoch_050_grid.png
grid_paths = sorted(glob.glob(os.path.join(gen_dir, "epoch_*_grid.png")))
grid_epochs = []
for p in grid_paths:
    base = os.path.basename(p)  # epoch_050_grid.png
    ep = int(base.split("_")[1])  # "050" -> 50
    grid_epochs.append(ep)
grid_epochs = sorted(set(grid_epochs))

# Expected ranges based on your setup:
# - checkpoint epochs are saved at loop index: 0..49 (and should include 50 if you forced it)
# - grids are saved as epoch+1: 1..50
expected_ckpt = list(range(0, 51))   # 0..50
expected_grid = list(range(1, 51))   # 1..50

missing_gen = [e for e in expected_ckpt if e not in set(gen_epochs)]
missing_disc = [e for e in expected_ckpt if e not in set(disc_epochs)]
missing_grids = [e for e in expected_grid if e not in set(grid_epochs)]

print("="*80)
print("🔎 FULL TRAINING CHECK VERIFICATION")
print("="*80)

print(f"📁 Checkpoint dir: {ckpt_dir}")
print(f"📁 Generated grids dir: {gen_dir}")
print("-"*80)

print("🧠 training_state.pth:")
print(f"  exists: {os.path.exists(state_path)}")
print(f"  last epoch stored: {state_epoch}")

print("-"*80)

print("🧬 Generator checkpoints:")
print(f"  epochs present: {gen_epochs[0]} .. {gen_epochs[-1]} (count={len(gen_epochs)})")
print(f"  generator_epoch_50.pth exists: {os.path.exists(os.path.join(ckpt_dir, 'generator_epoch_50.pth'))}")
print(f"  missing generator epochs (0..50): {missing_gen if missing_gen else 'None'}")

print("-"*80)

print("🧬 Discriminator checkpoints:")
print(f"  epochs present: {disc_epochs[0]} .. {disc_epochs[-1]} (count={len(disc_epochs)})")
print(f"  discriminator_epoch_50.pth exists: {os.path.exists(os.path.join(ckpt_dir, 'discriminator_epoch_50.pth'))}")
print(f"  missing discriminator epochs (0..50): {missing_disc if missing_disc else 'None'}")

print("-"*80)

print("🖼️ Generated image grids:")
print(f"  grid epochs present: {grid_epochs[0]} .. {grid_epochs[-1]} (count={len(grid_epochs)})")
print(f"  epoch_050_grid.png exists: {os.path.exists(os.path.join(gen_dir, 'epoch_050_grid.png'))}")
print(f"  missing grid epochs (1..50): {missing_grids if missing_grids else 'None'}")

print("="*80)

# Extra: show last few checkpoint filenames for sanity
print("📌 Last few generator checkpoint files:")
for f in gen_files[-5:]:
    print("   ", os.path.basename(f))

print("📌 Last few discriminator checkpoint files:")
for f in disc_files[-5:]:
    print("   ", os.path.basename(f))

print("="*80)

In [ ]:

import os, pandas as pd, torch

# Paths
loss_csv = "/kaggle/working/loss_history.csv"
ckpt_dir = "/kaggle/working/checkpoints"
gen_dir  = "/kaggle/working/generated_images"

# Loss CSV checks
df = pd.read_csv(loss_csv)
print("✅ loss_history.csv loaded:", df.shape)
print("Last 5 rows:\n", df.tail())

# Basic loss trend checks
print("\n📉 g_loss stats:", df["g_loss"].min(), "→", df["g_loss"].max())
print("📉 d_loss stats:", df["d_loss"].min(), "→", df["d_loss"].max())

# Check training_state
state = torch.load(os.path.join(ckpt_dir, "training_state.pth"), map_location="cpu")
print("\n✅ training_state.pth:")
print("  epoch:", state["epoch"])

# Check final checkpoints
print("\n✅ Final checkpoints exist?")
print(" generator_epoch_50:", os.path.exists(os.path.join(ckpt_dir, "generator_epoch_50.pth")))
print(" discriminator_epoch_50:", os.path.exists(os.path.join(ckpt_dir, "discriminator_epoch_50.pth")))

# Check final grid exists
print("\n✅ Final generated grid exists?")
print(" epoch_050_grid.png:", os.path.exists(os.path.join(gen_dir, "epoch_050_grid.png")))

In [ ]:

import os
import matplotlib.pyplot as plt
from PIL import Image

gen_dir = "/kaggle/working/generated_images"
epochs_to_show = [10, 20, 30, 40, 50]

fig, axes = plt.subplots(1, len(epochs_to_show), figsize=(22, 4))
for ax, ep in zip(axes, epochs_to_show):
    path = os.path.join(gen_dir, f"epoch_{ep:03d}_grid.png")
    img = Image.open(path)
    ax.imshow(img)
    ax.set_title(f"Epoch {ep}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:

from PIL import Image
import matplotlib.pyplot as plt

img_path = "/kaggle/working/generated_images/epoch_050_grid.png"
img = Image.open(img_path)

plt.figure(figsize=(10,10))
plt.imshow(img)
plt.axis("off")
plt.title("Epoch 50 Generated Grid")
plt.show()

**
📊 Notebook 05 - Visualization (Loss Curves + Samples)**

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

loss_history_path = "/kaggle/working/loss_history.csv"
df = pd.read_csv(loss_history_path)

plt.figure(figsize=(10,5))
plt.plot(df["epoch"], df["g_loss"], label="Generator Loss (G)")
plt.plot(df["epoch"], df["d_loss"], label="Discriminator Loss (D)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss Curves")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:

from PIL import Image
import os
import matplotlib.pyplot as plt

gen_dir = "/kaggle/working/generated_images"
epochs_to_show = [10, 20, 30, 40, 50]

fig, axes = plt.subplots(1, len(epochs_to_show), figsize=(22, 4))
for ax, ep in zip(axes, epochs_to_show):
    path = os.path.join(gen_dir, f"epoch_{ep:03d}_grid.png")
    img = Image.open(path)
    ax.imshow(img)
    ax.set_title(f"Epoch {ep}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:

import os, glob
import imageio

gen_dir = "/kaggle/working/generated_images"
frames = []
epoch_list = list(range(1, 51))  # 1..50

for ep in epoch_list:
    frame_path = os.path.join(gen_dir, f"epoch_{ep:03d}_grid.png")
    if os.path.exists(frame_path):
        frames.append(imageio.imread(frame_path))

gif_path = "/kaggle/working/training_progress.gif"
imageio.mimsave(gif_path, frames, fps=5)

print("✅ Saved GIF to:", gif_path)

**🧪 Notebook 06 - Evaluation (Generate Faces using trained Generator)**

In [ ]:

import os
import torch
import torchvision.utils as vutils
import matplotlib.pyplot as plt
from PIL import Image

# Paths
ckpt_dir = "/kaggle/working/checkpoints"
gen_w_path = os.path.join(ckpt_dir, "generator_epoch_50.pth")

assert os.path.exists(gen_w_path), f"Generator weights not found: {gen_w_path}"
print("✅ Found:", gen_w_path)

# Recreate Generator architecture (must match Notebook 03 exactly)
# (If you still have Generator class defined above in Notebook 05/04, you can skip redefining.)
LATENT_DIM = 100
CHANNELS = 3
IMAGE_SIZE = 64
NGF = 512

class Generator(torch.nn.Module):
    def __init__(self, latent_dim=100, img_channels=3, feature_maps=512):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.ConvTranspose2d(latent_dim, feature_maps, kernel_size=4, stride=1, padding=0, bias=False),
            torch.nn.BatchNorm2d(feature_maps),
            torch.nn.ReLU(True),

            torch.nn.ConvTranspose2d(feature_maps, feature_maps // 2, kernel_size=4, stride=2, padding=1, bias=False),
            torch.nn.BatchNorm2d(feature_maps // 2),
            torch.nn.ReLU(True),

            torch.nn.ConvTranspose2d(feature_maps // 2, feature_maps // 4, kernel_size=4, stride=2, padding=1, bias=False),
            torch.nn.BatchNorm2d(feature_maps // 4),
            torch.nn.ReLU(True),

            torch.nn.ConvTranspose2d(feature_maps // 4, feature_maps // 8, kernel_size=4, stride=2, padding=1, bias=False),
            torch.nn.BatchNorm2d(feature_maps // 8),
            torch.nn.ReLU(True),

            torch.nn.ConvTranspose2d(feature_maps // 8, img_channels, kernel_size=4, stride=2, padding=1, bias=False),
            torch.nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
G = Generator(latent_dim=LATENT_DIM, img_channels=CHANNELS, feature_maps=NGF).to(DEVICE)

# Load weights
G.load_state_dict(torch.load(gen_w_path, map_location=DEVICE))
G.eval()
print("✅ Generator loaded and set to eval()")

# Generate new samples
with torch.no_grad():
    noise = torch.randn(64, LATENT_DIM, 1, 1, device=DEVICE)
    fake = G(noise).detach()  # [-1,1]

def denorm(x):
    return (x + 1) / 2  # -> [0,1]

fake = denorm(fake).cpu()

# Make grid
grid = vutils.make_grid(fake, nrow=8, padding=2, normalize=False)
plt.figure(figsize=(10,10))
plt.axis("off")
plt.imshow(grid.permute(1,2,0).numpy())
plt.title("✅ Generated Anime Faces (Epoch 50 Generator)")
plt.show()

# Save
out_path = "/kaggle/working/generated_faces_eval_epoch50.png"
plt.imsave(out_path, grid.permute(1,2,0).numpy())
print("✅ Saved:", out_path)


Notebook 07 - Generate More Faces (Final Samples)

In [ ]:

import os
import torch
import torchvision.utils as vutils
import matplotlib.pyplot as plt
from PIL import Image
import glob

# Paths
ckpt_dir = "/kaggle/working/checkpoints"
gen_w_path = os.path.join(ckpt_dir, "generator_epoch_50.pth")

# Recreate Generator (must match Notebook 03)
LATENT_DIM = 100
CHANNELS = 3
NGF = 512

class Generator(torch.nn.Module):
    def __init__(self, latent_dim=100, img_channels=3, feature_maps=512):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.ConvTranspose2d(latent_dim, feature_maps, kernel_size=4, stride=1, padding=0, bias=False),
            torch.nn.BatchNorm2d(feature_maps),
            torch.nn.ReLU(True),

            torch.nn.ConvTranspose2d(feature_maps, feature_maps // 2, kernel_size=4, stride=2, padding=1, bias=False),
            torch.nn.BatchNorm2d(feature_maps // 2),
            torch.nn.ReLU(True),

            torch.nn.ConvTranspose2d(feature_maps // 2, feature_maps // 4, kernel_size=4, stride=2, padding=1, bias=False),
            torch.nn.BatchNorm2d(feature_maps // 4),
            torch.nn.ReLU(True),

            torch.nn.ConvTranspose2d(feature_maps // 4, feature_maps // 8, kernel_size=4, stride=2, padding=1, bias=False),
            torch.nn.BatchNorm2d(feature_maps // 8),
            torch.nn.ReLU(True),

            torch.nn.ConvTranspose2d(feature_maps // 8, img_channels, kernel_size=4, stride=2, padding=1, bias=False),
            torch.nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
G = Generator(latent_dim=LATENT_DIM, img_channels=CHANNELS, feature_maps=NGF).to(DEVICE)

# Load weights
assert os.path.exists(gen_w_path), f"Missing weights: {gen_w_path}"
G.load_state_dict(torch.load(gen_w_path, map_location=DEVICE))
G.eval()
print("✅ Loaded Generator (epoch 50)")

def denorm(x):
    return (x + 1) / 2  # [-1,1] -> [0,1]

# Generate final samples
out_dir = "/kaggle/working/final_samples"
os.makedirs(out_dir, exist_ok=True)

N = 256  # total images to generate (4 batches of 64)
batch_size = 64

all_grids = []
with torch.no_grad():
    for b in range(N // batch_size):
        noise = torch.randn(batch_size, LATENT_DIM, 1, 1, device=DEVICE)
        fake = G(noise)
        fake = denorm(fake).clamp(0, 1).cpu()  # [0,1]

        grid = vutils.make_grid(fake, nrow=8, padding=2, normalize=False)
        all_grids.append(grid)

        # Save each batch grid
        grid_path = os.path.join(out_dir, f"final_batch_{b+1:02d}_grid.png")
        plt.figure(figsize=(10,10))
        plt.axis("off")
        plt.imshow(grid.permute(1,2,0).numpy())
        plt.tight_layout()
        plt.savefig(grid_path, dpi=150, bbox_inches="tight")
        plt.close()
        print("✅ Saved:", grid_path)

# Create one big combined grid image (stacks batch grids vertically)
# Convert grids to numpy and stitch
combined = torch.cat(all_grids, dim=1)  # stack along height dimension
combined_np = combined.permute(1,2,0).numpy()

combined_path = os.path.join(out_dir, "final_all_batches_grid.png")
plt.figure(figsize=(12, 24))
plt.axis("off")
plt.imshow(combined_np)
plt.tight_layout()
plt.savefig(combined_path, dpi=150, bbox_inches="tight")
plt.close()

print("\n🎉 Final combined grid saved to:", combined_path)

In [ ]:

import torch

ckpt_dir = "/kaggle/working/checkpoints"
gen_w_path = f"{ckpt_dir}/generator_epoch_50.pth"

# Generator definition must match training exactly
LATENT_DIM = 100
CHANNELS = 3
NGF = 512

class Generator(torch.nn.Module):
    def __init__(self, latent_dim=100, img_channels=3, feature_maps=512):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.ConvTranspose2d(latent_dim, feature_maps, 4, 1, 0, bias=False),
            torch.nn.BatchNorm2d(feature_maps),
            torch.nn.ReLU(True),

            torch.nn.ConvTranspose2d(feature_maps, feature_maps // 2, 4, 2, 1, bias=False),
            torch.nn.BatchNorm2d(feature_maps // 2),
            torch.nn.ReLU(True),

            torch.nn.ConvTranspose2d(feature_maps // 2, feature_maps // 4, 4, 2, 1, bias=False),
            torch.nn.BatchNorm2d(feature_maps // 4),
            torch.nn.ReLU(True),

            torch.nn.ConvTranspose2d(feature_maps // 4, feature_maps // 8, 4, 2, 1, bias=False),
            torch.nn.BatchNorm2d(feature_maps // 8),
            torch.nn.ReLU(True),

            torch.nn.ConvTranspose2d(feature_maps // 8, img_channels, 4, 2, 1, bias=False),
            torch.nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
G = Generator(LATENT_DIM, CHANNELS, NGF).to(device)
G.load_state_dict(torch.load(gen_w_path, map_location=device))
G.eval()

# Script it for portable inference
scripted = torch.jit.trace(G, torch.randn(1, LATENT_DIM, 1, 1, device=device))
out_model_path = "/kaggle/working/generator_epoch_50_scripted.pt"
scripted.save(out_model_path)

print("✅ Saved scripted inference model to:", out_model_path)

In [ ]:

import os
import torch

ckpt_dir = "/kaggle/working/checkpoints"
disc_w_path = os.path.join(ckpt_dir, "discriminator_epoch_50.pth")
assert os.path.exists(disc_w_path), f"Missing: {disc_w_path}"

# Must match your Notebook 03 Discriminator architecture
LATENT_DIM = 100
CHANNELS = 3
NDF = 512
IMAGE_SIZE = 64

class Discriminator(torch.nn.Module):
    def __init__(self, img_channels=3, feature_maps=512):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Conv2d(img_channels, feature_maps // 8, 4, 2, 1, bias=False),
            torch.nn.LeakyReLU(0.2, inplace=True),

            torch.nn.Conv2d(feature_maps // 8, feature_maps // 4, 4, 2, 1, bias=False),
            torch.nn.BatchNorm2d(feature_maps // 4),
            torch.nn.LeakyReLU(0.2, inplace=True),

            torch.nn.Conv2d(feature_maps // 4, feature_maps // 2, 4, 2, 1, bias=False),
            torch.nn.BatchNorm2d(feature_maps // 2),
            torch.nn.LeakyReLU(0.2, inplace=True),

            torch.nn.Conv2d(feature_maps // 2, feature_maps, 4, 2, 1, bias=False),
            torch.nn.BatchNorm2d(feature_maps),
            torch.nn.LeakyReLU(0.2, inplace=True),

            torch.nn.Conv2d(feature_maps, 1, 4, 1, 0, bias=False),
            torch.nn.Sigmoid()
        )

    def forward(self, x):
        out = self.net(x)          # (N,1,1,1)
        return out.view(out.size(0), -1)  # (N,1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
D = Discriminator(img_channels=CHANNELS, feature_maps=NDF).to(device)
D.load_state_dict(torch.load(disc_w_path, map_location=device))
D.eval()

# Script/trace
dummy = torch.randn(1, CHANNELS, IMAGE_SIZE, IMAGE_SIZE, device=device)
scriptedD = torch.jit.trace(D, dummy)

out_model_path = "/kaggle/working/discriminator_epoch_50_scripted.pt"
scriptedD.save(out_model_path)
print("✅ Saved scripted Discriminator to:", out_model_path)